# LLM 기반 알라딘 사용자 리뷰 분석

Google Play Store의 알라딘 앱 리뷰를 수집하고, 사전학습된 LLM으로 부정적 리뷰의 UI/UX 관련 여부와 주요 불편 유형을 구조화하는 탐색적 분석 프로젝트입니다. 기업 문제 탐색 활동 중 리뷰로 관찰 가능한 사용자 경험에 초점을 맞추었습니다. 브랜드 문제 전체의 검증이나 실제 서비스 개선 효과 측정은 범위에 포함하지 않습니다.

- **활동:** AI POC 클리닉
- **수행 기간:** 2026.08.05–2026.09.11
- **팀 규모:** 3명 / **본인 역할:** 데이터 분석
- **분석 질문:** 선정한 부정적 리뷰에서 어떤 UI/UX 불편이 반복되는가?
- **방법:** 관련도 순 리뷰 수집 → 별점 1~2점 선별 → few-shot 프롬프트 기반 단일 범주 분류 → JSON 구조화 → 집계 및 CSV 저장

별도의 모델 학습·파인튜닝이나 외부 LLM API 호출은 없으며, Transformers로 Qwen을 직접 로드해 추론합니다.

## 실행 전 확인

Python 기반 Jupyter/Colab 환경과 CUDA GPU가 필요합니다. 코드는 입력 텐서를 `cuda`로 이동하고 4비트 양자화에 `bitsandbytes`를 사용합니다. NanumGothic 폰트 설정은 폰트를 설치하지 않으며, 이 노트북에는 실제 차트 생성 코드가 없습니다. 의존성 버전이 고정되어 있지 않고 리뷰가 실시간 수집되므로 재실행 결과는 원본과 달라질 수 있습니다. 공개 데이터라 해도 리뷰 원문·사용자명 등을 포함한 CSV는 재배포 조건과 식별정보를 검토한 뒤 공유해야 합니다.


## 1. 의존성 설치 및 라이브러리 준비
다음 셀은 리뷰 수집, 데이터 처리, 양자화 모델 추론에 필요한 라이브러리를 준비합니다. `!pip`는 노트북에서 사용하는 설치 명령입니다. Matplotlib과 Seaborn은 가져오지만 이 파일에서는 차트를 생성하지 않습니다.


In [ ]:
!pip install google-play-scraper
!pip install -q -U transformers bitsandbytes accelerate tqdm

from google_play_scraper import Sort, reviews
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
import json


## 2. Google Play Store 리뷰 수집
알라딘 앱 식별자 `kr.co.aladin.third_shop`에 대해 한국어·한국 지역의 **관련도 순** 리뷰 500개를 요청하고 DataFrame으로 변환합니다. 500개는 요청 건수이며 실제 수집 결과는 실행 시 확인해야 합니다. 최신순 또는 무작위 표본이 아닙니다.


In [ ]:
result, _ = reviews(
    'kr.co.aladin.third_shop',
    lang='ko',
    country='kr',
    sort=Sort.MOST_RELEVANT,
    count=500
)

df = pd.DataFrame(result)


### 수집 데이터의 기초 통계 확인
`describe()`로 별점 등 수집 데이터의 기초 통계를 확인합니다. 이 단계는 분류 모델의 성능 평가가 아닙니다.


In [ ]:
df.describe()


## 3. 4비트 양자화 Qwen 모델 로드
`Qwen/Qwen2.5-7B-Instruct`를 NF4 4비트 양자화, FP16 연산, 이중 양자화 설정으로 로드합니다. `device_map="auto"`로 모델 배치를 자동 지정합니다. 양자화는 모델 로드 시 설정이며 모델을 새로 학습하는 과정이 아닙니다.


In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


## 4. Few-shot 프롬프트 기반 리뷰 분류 함수
한국어 리뷰를 입력하고 영어 JSON으로 UI/UX 여부, 단일 토픽, 분류 이유를 생성합니다. 문맥이 다양한 리뷰를 사전 정의 범주로 구조화하는 데 LLM을 사용하며, 키워드 분류 대비 성능 우위는 이 파일에서 평가하지 않습니다.

| 토픽 | 프롬프트의 분류 기준 |
|---|---|
| `SEARCH_NAVIGATION` | 책·중고 상품 탐색, 숨겨진 탭, 복잡한 탐색 깊이 |
| `LAYOUT_CLUTTER` | 과밀한 텍스트·화면, 가독성 저하, 시각적 혼잡 |
| `INTERACTION_AFFORDANCE` | 버튼 무반응, 클릭 가능성 인지, 터치 영역 문제 |
| `CART_CHECKOUT_UX` | 장바구니 선택 흐름, 뒤로 이동 시 선택 유지 문제 |
| `NON_UI_UX` | 앱 충돌, 서버 지연, 결제 인증 오류, 배송 문제 |

프롬프트는 UI/UX 문제를 하나의 토픽으로 분류하도록 지시합니다. 여러 문제가 섞인 리뷰에서는 일부 정보가 축약될 수 있고, 반응 없음과 시스템 지연·충돌의 경계도 모호할 수 있습니다. 예시는 프롬프트에 포함된 설명용 예시이며 실제 수집 리뷰라고 제시하지 않습니다.

`do_sample=False`로 비샘플링 생성을 사용합니다. 생성 토큰은 최대 120개이며, 분류 이유는 검토용 모델 출력이지 분류의 정확성을 보장하는 근거가 아닙니다.


In [ ]:
def analyze_review_with_topics(review_text):
    prompt = f"""You are an expert UX/UI Data Analyst. Analyze the Korean user review for the 'Aladin' app.
Determine if the review describes a UI/UX issue (is_ui_ux_issue: 1) or non-UI/UX issue (is_ui_ux_issue: 0).
If it is a UI/UX issue, classify it into EXACTLY ONE of the following controlled UI/UX topics:
- SEARCH_NAVIGATION: Hard to find books/used items, hidden tabs, complex navigation depth.
- LAYOUT_CLUTTER: Overcrowded text, cluttered screen layout, poor readability, visual noise.
- INTERACTION_AFFORDANCE: Unresponsive buttons, unclickable UI design, poor touch targets.
- CART_CHECKOUT_UX: Confusing cart selection flow, losing item selection on back button.
- NON_UI_UX: System crash, server lag, payment authentication error, delivery issues.

Respond ONLY with a valid JSON object in English.

### FEW-SHOT EXAMPLES:

Review: "중고책 어디서 찾는지 탭이 숨겨져 있고 검색 결과에서 매번 다시 선택해야 해서 짜증납니다."
JSON Response:
{{
  "is_ui_ux_issue": 1,
  "ui_ux_topic": "SEARCH_NAVIGATION",
  "primary_reason": "Hidden used-book tabs and deep search navigation depth."
}}

Review: "화면에 글씨랑 이벤트 배너가 너무 많아서 눈이 아프고 뭐가 뭔지 하나도 안 보여요."
JSON Response:
{{
  "is_ui_ux_issue": 1,
  "ui_ux_topic": "LAYOUT_CLUTTER",
  "primary_reason": "Densely packed text and high visual clutter causing eye strain."
}}

Review: "중고 상품 버튼처럼 생기지 않아서 한참 찾았네요. 누르는 건지도 몰랐음."
JSON Response:
{{
  "is_ui_ux_issue": 1,
  "ui_ux_topic": "INTERACTION_AFFORDANCE",
  "primary_reason": "Poor button affordance; CTA element looks like static text."
}}

Review: "서버가 터진 건지 결제창 넘어갈 때 계속 앱이 꺼집니다."
JSON Response:
{{
  "is_ui_ux_issue": 0,
  "ui_ux_topic": "NON_UI_UX",
  "primary_reason": "System crash during payment transition."
}}

### INPUT REVIEW:
Review: "{review_text}"

JSON Response:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


## 5. 부정적 리뷰 선별 및 반복 추론
별점 `score <= 2`인 리뷰를 선별하고 한 건씩 추론합니다. 응답에서 중괄호 구간을 추출해 JSON으로 파싱하고, 원본 리뷰·별점·사용자명·작성일과 결합해 `df_analyzed`를 만듭니다.


In [ ]:
plt.rcParams['font.sans-serif'] = ['NanumGothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False


df_neg = df[df['score'] <= 2].copy().reset_index(drop=True)


results = []

print(f"총 {len(df_neg)}건의 부정 리뷰에 대해 LLM 문맥 분석을 시작합니다.")

for idx, row in tqdm(df_neg.iterrows(), total=len(df_neg), desc="LLM Analyzing"):
    review_text = str(row['content'])


    raw_out = analyze_review_with_topics(review_text)


    try:
        json_str = raw_out[raw_out.find('{'):raw_out.rfind('}')+1]
        parsed = json.loads(json_str)
    except Exception as e:
        parsed = {
            "is_ui_ux_issue": 0,
            "ui_ux_topic": "NON_UI_UX",
            "primary_reason": "JSON Parsing Fallback"
        }


    parsed['userName'] = row.get('userName', '')
    parsed['score'] = row['score']
    parsed['content'] = review_text
    parsed['at'] = row.get('at', '')

    results.append(parsed)


df_analyzed = pd.DataFrame(results)


## 6. UI/UX 분류 비중 및 세부 토픽 집계
전체 분석 대상 리뷰를 분모로 UI/UX 분류 비율을 계산합니다. 이어 UI/UX로 분류된 리뷰만을 분모로 토픽별 건수와 비율을 계산합니다. 두 비율은 분모가 다릅니다. 출력 문구의 'UI/UX 원인'은 원본 코드의 명칭이며, 실제 의미는 **모델이 UI/UX 관련으로 분류했다는 것**이지 원인 입증이 아닙니다.


In [ ]:
total_neg = len(df_analyzed)
ui_ux_df = df_analyzed[df_analyzed['is_ui_ux_issue'] == 1]
ui_ux_count = len(ui_ux_df)
ui_ux_ratio = (ui_ux_count / total_neg) * 100

print("=" * 50)
print(f"[1. LLM 이진 분류 결과]")
print(f"• 전체 분석 부정 리뷰: {total_neg}건")
print(f"• UI/UX 원인 부정 리뷰: {ui_ux_count}건 ({ui_ux_ratio:.1f}%)")
print("=" * 50)


print("\n[2. UI/UX 세부 문제 토픽 분포 (N={})]".format(ui_ux_count))
topic_counts = ui_ux_df['ui_ux_topic'].value_counts()
topic_ratios = ui_ux_df['ui_ux_topic'].value_counts(normalize=True) * 100

for topic, count in topic_counts.items():
    pct = topic_ratios[topic]
    print(f"• {topic:<23}: {count:>3}건 ({pct:>5.1f}%)")


## 7. 분석 결과 저장 및 샘플 확인
전체 분석 결과를 `aladin_llm_topic_analysis.csv`로 저장하고, UI/UX로 분류한 리뷰의 첫 5건을 분류 이유와 함께 표시합니다. `head(5)`는 무작위 표본이나 전체 결과 검증이 아닙니다. CSV에는 사용자명·리뷰 원문·작성일이 포함되므로 공개 저장소에 자동으로 추가하지 않는 것을 권장합니다.


In [ ]:
df_analyzed.to_csv('aladin_llm_topic_analysis.csv', index=False, encoding='utf-8-sig')
print("파일 저장 완료: aladin_llm_topic_analysis.csv")


print("\n[LLM UI/UX 토픽 분류 샘플 5건]")
sample_cols = ['score', 'ui_ux_topic', 'primary_reason', 'content']
df_analyzed[df_analyzed['is_ui_ux_issue'] == 1][sample_cols].head(5)


## 결과

- 수집 데이터 기초 통계의 리뷰 수: **500개**
- 분석 대상 부정적 리뷰: **350개**
- UI/UX 관련으로 분류된 리뷰: **111개 / 350개 (31.7%)**

| UI/UX 토픽 | 리뷰 수 | UI/UX 분류 111개 내 비중 |
|---|---:|---:|
| INTERACTION_AFFORDANCE | 46 | 41.4% |
| SEARCH_NAVIGATION | 37 | 33.3% |
| CART_CHECKOUT_UX | 16 | 14.4% |
| LAYOUT_CLUTTER | 12 | 10.8% |

토픽별 1점 리뷰 비율은 이 파일의 집계 코드·저장 출력에 나타나지 않아 포함하지 않았습니다.

## 한계

- 관련도 순 리뷰 중 낮은 별점만 선택한 표본으로 전체 사용자의 불편 비율을 추정할 수 없습니다.
- 분류 컬럼은 사람의 판단 하에 임의로 설정되었습니다.
- Few-shot 단일 토픽 분류는 복합적인 리뷰를 축약할 수 있습니다.
- 파싱 실패 처리와 라벨 검증 부재가 집계에 영향을 줄 수 있습니다. 실패 건수는 저장된 출력에서 확인되지 않습니다.
